In [21]:
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd

In [22]:
def read_str(data: bytes, start: int, length: int) -> str:
    """
    Lee una porción de bytes y la convierte a string ASCII.

    Parameters
    ----------
    data : bytes
        Bloque de bytes leído desde el archivo BDF.
    start : int
        Posición inicial desde donde leer.
    length : int
        Cantidad de bytes a leer.

    Returns
    -------
    str
        Texto decodificado y limpiado.
    """
    return data[start : start + length].decode("ascii", errors="ignore").strip()

In [23]:
def read_bdf_fixed_header(file_path: Path) -> Dict[str, Any]:
    """
    Lee el header fijo de 256 bytes de un archivo BDF.

    Parameters
    ----------
    file_path : Path
        Ruta del archivo .bdf.

    Returns
    -------
    Dict[str, Any]
        Diccionario con información principal del header fijo.
    """
    with open(file_path, "rb") as file:
        fixed_header: bytes = file.read(256)

    header_info: Dict[str, Any] = {
        "version": read_str(fixed_header, 0, 8),
        "patient_id": read_str(fixed_header, 8, 80),
        "recording_id": read_str(fixed_header, 88, 80),
        "start_date": read_str(fixed_header, 168, 8),
        "start_time": read_str(fixed_header, 176, 8),
        "header_bytes": int(read_str(fixed_header, 184, 8)),
        "reserved": read_str(fixed_header, 192, 44),
        "num_records": int(read_str(fixed_header, 236, 8)),
        "record_duration": float(read_str(fixed_header, 244, 8)),
        "num_channels": int(read_str(fixed_header, 252, 4)),
    }

    return header_info

In [24]:
def read_bdf_channel_headers(file_path: Path) -> pd.DataFrame:
    """
    Lee el header de canales de un archivo BDF.

    Parameters
    ----------
    file_path : Path
        Ruta del archivo .bdf.

    Returns
    -------
    pd.DataFrame
        DataFrame con la metadata de cada canal.
    """
    fixed_header_info: Dict[str, Any] = read_bdf_fixed_header(file_path)

    header_bytes: int = fixed_header_info["header_bytes"]
    num_channels: int = fixed_header_info["num_channels"]

    with open(file_path, "rb") as file:
        full_header: bytes = file.read(header_bytes)

    signal_header_start: int = 256

    field_sizes: List[Tuple[str, int]] = [
        ("label", 16),
        ("transducer", 80),
        ("physical_dimension", 8),
        ("physical_min", 8),
        ("physical_max", 8),
        ("digital_min", 8),
        ("digital_max", 8),
        ("prefiltering", 80),
        ("samples_per_record", 8),
        ("reserved", 32),
    ]

    field_starts: Dict[str, int] = {}

    current_start: int = signal_header_start

    for field_name, field_size in field_sizes:
        field_starts[field_name] = current_start
        current_start += field_size * num_channels

    channel_rows: List[Dict[str, Any]] = []

    participant: str = file_path.stem

    for channel_index in range(num_channels):
        row: Dict[str, Any] = {
            "participant": participant,
            "channel_index": channel_index + 1,
            "record_duration": fixed_header_info["record_duration"],
            "num_records": fixed_header_info["num_records"],
        }

        for field_name, field_size in field_sizes:
            start: int = field_starts[field_name] + channel_index * field_size
            value: str = read_str(full_header, start, field_size)
            row[field_name] = value

        row["samples_per_record"] = int(row["samples_per_record"])
        row["sampling_frequency"] = row["samples_per_record"] / row["record_duration"]

        channel_rows.append(row)

    return pd.DataFrame(channel_rows)

In [25]:
dataset_dir: Path = Path("../dataset/raw/bdf")

bdf_files: List[Path] = sorted(dataset_dir.glob("s*.bdf"))

print(f"Cantidad de archivos BDF encontrados: {len(bdf_files)}")

for file_path in bdf_files:
    print(file_path.name)

Cantidad de archivos BDF encontrados: 32
s01.bdf
s02.bdf
s03.bdf
s04.bdf
s05.bdf
s06.bdf
s07.bdf
s08.bdf
s09.bdf
s10.bdf
s11.bdf
s12.bdf
s13.bdf
s14.bdf
s15.bdf
s16.bdf
s17.bdf
s18.bdf
s19.bdf
s20.bdf
s21.bdf
s22.bdf
s23.bdf
s24.bdf
s25.bdf
s26.bdf
s27.bdf
s28.bdf
s29.bdf
s30.bdf
s31.bdf
s32.bdf


In [26]:
all_channel_headers: List[pd.DataFrame] = []

for file_path in bdf_files:
    print(f"Leyendo: {file_path.name}")
    participant_header_df: pd.DataFrame = read_bdf_channel_headers(file_path)
    all_channel_headers.append(participant_header_df)

all_samples_per_record_df: pd.DataFrame = pd.concat(
    all_channel_headers,
    ignore_index=True,
)

all_samples_per_record_df.head()

Leyendo: s01.bdf
Leyendo: s02.bdf
Leyendo: s03.bdf
Leyendo: s04.bdf
Leyendo: s05.bdf
Leyendo: s06.bdf
Leyendo: s07.bdf
Leyendo: s08.bdf
Leyendo: s09.bdf
Leyendo: s10.bdf
Leyendo: s11.bdf
Leyendo: s12.bdf
Leyendo: s13.bdf
Leyendo: s14.bdf
Leyendo: s15.bdf
Leyendo: s16.bdf
Leyendo: s17.bdf
Leyendo: s18.bdf
Leyendo: s19.bdf
Leyendo: s20.bdf
Leyendo: s21.bdf
Leyendo: s22.bdf
Leyendo: s23.bdf
Leyendo: s24.bdf
Leyendo: s25.bdf
Leyendo: s26.bdf
Leyendo: s27.bdf
Leyendo: s28.bdf
Leyendo: s29.bdf
Leyendo: s30.bdf
Leyendo: s31.bdf
Leyendo: s32.bdf


,participant,channel_index,record_duration,num_records,label,transducer,physical_dimension,physical_min,physical_max,digital_min,digital_max,prefiltering,samples_per_record,reserved,sampling_frequency
0,s01,1,1.0,3869,Fp1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON,512.0
1,s01,2,1.0,3869,AF3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON,512.0
2,s01,3,1.0,3869,F7,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON,512.0
3,s01,4,1.0,3869,F3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON,512.0
4,s01,5,1.0,3869,FC1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON,512.0


In [27]:
samples_summary_df: pd.DataFrame = all_samples_per_record_df[
    [
        "participant",
        "channel_index",
        "label",
        "samples_per_record",
        "record_duration",
        "sampling_frequency",
    ]
]

samples_summary_df

,participant,channel_index,label,samples_per_record,record_duration,sampling_frequency
0,s01,1,Fp1,512,1.0,512.0
1,s01,2,AF3,512,1.0,512.0
2,s01,3,F7,512,1.0,512.0
3,s01,4,F3,512,1.0,512.0
4,s01,5,FC1,512,1.0,512.0
...,...,...,...,...,...,...
1535,s32,45,Resp,512,1.0,512.0
1536,s32,46,Plet,512,1.0,512.0
1537,s32,47,Temp,512,1.0,512.0
1538,s32,48,,512,1.0,512.0


In [28]:
unique_samples_per_record = samples_summary_df["samples_per_record"].unique()
unique_sampling_frequency = samples_summary_df["sampling_frequency"].unique()

print("Valores únicos de samples_per_record:")
print(unique_samples_per_record)

print("\nValores únicos de sampling_frequency:")
print(unique_sampling_frequency)

Valores únicos de samples_per_record:
[512]

Valores únicos de sampling_frequency:
[512.]


In [ ]:
validation_by_participant_df: pd.DataFrame = (
    all_samples_per_record_df.groupby("participant")
    .agg(
        num_channels=("channel_index", "count"),
        record_duration=("record_duration", "first"),
        num_records=("num_records", "first"),
        unique_samples_per_record=(
            "samples_per_record",
            lambda values: sorted(values.unique()),
        ),
    )
    .reset_index()
)

validation_by_participant_df["total_duration_seconds"] = (
    validation_by_participant_df["num_records"]
    * validation_by_participant_df["record_duration"]
)

validation_by_participant_df["total_duration_minutes"] = (
    validation_by_participant_df["total_duration_seconds"] / 60
)

validation_by_participant_df[
    "total_samples_per_channel"
] = validation_by_participant_df["num_records"] * validation_by_participant_df[
    "unique_samples_per_record"
].apply(
    lambda values: values[0]
)

validation_by_participant_df

,participant,num_channels,record_duration,num_records,unique_samples_per_record,total_duration_seconds,total_duration_minutes,total_samples_per_channel
0,s01,48,1.0,3869,[512],3869.0,64.483333,1980928
1,s02,48,1.0,3703,[512],3703.0,61.716667,1895936
2,s03,48,1.0,3885,[512],3885.0,64.750000,1989120
3,s04,48,1.0,3301,[512],3301.0,55.016667,1690112
4,s05,48,1.0,3916,[512],3916.0,65.266667,2004992
5,s06,48,1.0,3586,[512],3586.0,59.766667,1836032
6,s07,48,1.0,3571,[512],3571.0,59.516667,1828352
7,s08,48,1.0,3517,[512],3517.0,58.616667,1800704
8,s09,48,1.0,3767,[512],3767.0,62.783333,1928704
9,s10,48,1.0,3512,[512],3512.0,58.533333,1798144


In [30]:
all_same_samples_per_record: bool = (
    samples_summary_df["samples_per_record"].nunique() == 1
)
all_same_sampling_frequency: bool = (
    samples_summary_df["sampling_frequency"].nunique() == 1
)

print(
    f"¿Todos los canales tienen el mismo samples_per_record?: {all_same_samples_per_record}"
)
print(
    f"¿Todos los canales tienen la misma frecuencia de muestreo?: {all_same_sampling_frequency}"
)

¿Todos los canales tienen el mismo samples_per_record?: True
¿Todos los canales tienen la misma frecuencia de muestreo?: True


In [31]:
participant_summary_df: pd.DataFrame = (
    all_samples_per_record_df.groupby("participant")
    .agg(
        num_channels=("channel_index", "count"),
        record_duration=("record_duration", "first"),
        num_records=("num_records", "first"),
        unique_samples_per_record=(
            "samples_per_record",
            lambda values: sorted(values.unique()),
        ),
    )
    .reset_index()
)

# participant_summary_df
print(participant_summary_df.to_string(index=False))

participant  num_channels  record_duration  num_records unique_samples_per_record
        s01            48              1.0         3869                     [512]
        s02            48              1.0         3703                     [512]
        s03            48              1.0         3885                     [512]
        s04            48              1.0         3301                     [512]
        s05            48              1.0         3916                     [512]
        s06            48              1.0         3586                     [512]
        s07            48              1.0         3571                     [512]
        s08            48              1.0         3517                     [512]
        s09            48              1.0         3767                     [512]
        s10            48              1.0         3512                     [512]
        s11            48              1.0         4161                     [512]
        s12     

In [34]:
total_samples_sum: int = 0
total_bdf_bytes_sum: int = 0
total_ram_float32_bytes_sum: int = 0

for _, row in participant_summary_df.iterrows():
    num_channels: int = int(row["num_channels"])
    num_records: int = int(row["num_records"])
    samples_per_record: int = int(row["unique_samples_per_record"][0])

    total_samples: int = num_records * samples_per_record * num_channels

    total_bdf_bytes: int = (
        256
        + (256 * num_channels)
        + (total_samples * 3)
    )

    ram_float32_bytes: int = total_samples * 4

    total_samples_sum += total_samples
    total_bdf_bytes_sum += total_bdf_bytes
    total_ram_float32_bytes_sum += ram_float32_bytes

print("Total samples:", f"{total_samples_sum:,}")

print(
    "Estimated BDF size:",
    f"{total_bdf_bytes_sum / (1024 ** 2):.2f} MB",
    f"({total_bdf_bytes_sum / (1024 ** 3):.3f} GB)"
)

print(
    "RAM float32:",
    f"{total_ram_float32_bytes_sum / (1024 ** 2):.2f} MB",
    f"({total_ram_float32_bytes_sum / (1024 ** 3):.3f} GB)"
)

final_summary_display_df
# print(final_summary_display_df.to_string(index=False))

Total samples: 2,952,355,328
Estimated BDF size: 8447.14 MB (8.249 GB)
RAM float32: 11262.34 MB (10.998 GB)


NameError: name 'final_summary_display_df' is not defined

95084544 +
91004928 +
95477760 +
81125376 +
96239616 +
88129536 +
87760896 +
86433792 +
92577792 +
86310912 +
102260736 +
85966848 +
91201536 +
96755712 +
100073472 +
90513408 +
84688896 +
89202688 +
92106752 +
86237184 +
87269376 +
91398144 +
88498176 +
115802112 +
101253120 +
96165888 +
93339648 +
86679552 +
99840384 +
91924480 +
92702208 +
88435712